In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.datasets import cifar10
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import (confusion_matrix, classification_report,
                              precision_score, recall_score, f1_score, accuracy_score)

print("TensorFlow version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))

np.random.seed(42)
tf.random.set_seed(42)


In [ ]:
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

class_names = ['airplane', 'automobile', 'bird', 'cat', 'deer',
               'dog', 'frog', 'horse', 'ship', 'truck']

print("Training data shape:", x_train.shape)
print("Training labels shape:", y_train.shape)
print("Test data shape:", x_test.shape)
print("Test labels shape:", y_test.shape)


In [ ]:
unique, counts = np.unique(y_train, return_counts=True)

plt.figure(figsize=(10, 5))
plt.bar([class_names[i] for i in unique], counts, color='skyblue', edgecolor='black')
plt.title("Class Distribution in Training Set")
plt.xlabel("Class")
plt.ylabel("Number of Samples")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(10, 10))
for i in range(25):
    plt.subplot(5, 5, i + 1)
    plt.xticks([]); plt.yticks([]); plt.grid(False)
    plt.imshow(x_train[i])
    plt.xlabel(class_names[y_train[i][0]])
plt.tight_layout()
plt.suptitle("Sample CIFAR-10 Images", y=1.02)
plt.show()


In [ ]:
x_train_norm = x_train.astype('float32') / 255.0
x_test_norm = x_test.astype('float32') / 255.0

y_train_cat = to_categorical(y_train, 10)
y_test_cat = to_categorical(y_test, 10)

# Split a validation set from training data
val_split = 0.1
val_size = int(len(x_train_norm) * val_split)

x_val = x_train_norm[:val_size]
y_val = y_train_cat[:val_size]
x_train_final = x_train_norm[val_size:]
y_train_final = y_train_cat[val_size:]

print("Train:", x_train_final.shape, "Val:", x_val.shape, "Test:", x_test_norm.shape)


In [ ]:
def build_cnn(input_shape=(32, 32, 3), num_classes=10,
              filters=(32, 64, 128), kernel_size=(3, 3),
              pool_type='max', activation='relu'):
    """
    Builds a CNN with configurable filters, kernel size, and pooling type.
    pool_type: 'max' or 'avg'
    """
    PoolLayer = layers.MaxPooling2D if pool_type == 'max' else layers.AveragePooling2D

    model = models.Sequential(name=f"CNN_{pool_type}pool")
    model.add(layers.Input(shape=input_shape))

    # Block 1
    model.add(layers.Conv2D(filters[0], kernel_size, padding='same', activation=activation))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(filters[0], kernel_size, padding='same', activation=activation))
    model.add(PoolLayer(pool_size=(2, 2)))
    model.add(layers.Dropout(0.25))

    # Block 2
    model.add(layers.Conv2D(filters[1], kernel_size, padding='same', activation=activation))
    model.add(layers.BatchNormalization())
    model.add(layers.Conv2D(filters[1], kernel_size, padding='same', activation=activation))
    model.add(PoolLayer(pool_size=(2, 2)))
    model.add(layers.Dropout(0.25))

    # Block 3
    model.add(layers.Conv2D(filters[2], kernel_size, padding='same', activation=activation))
    model.add(layers.BatchNormalization())
    model.add(PoolLayer(pool_size=(2, 2)))
    model.add(layers.Dropout(0.3))

    # Fully Connected Head
    model.add(layers.Flatten())
    model.add(layers.Dense(256, activation=activation))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(num_classes, activation='softmax'))

    return model

cnn_model = build_cnn(pool_type='max')
cnn_model.summary()


In [ ]:
print(f"{'Layer Name':<30}{'Output Shape':<25}{'Trainable Params':<20}")
print("-" * 75)
total_trainable = 0
for layer in cnn_model.layers:
    params = layer.count_params()
    total_trainable += params if layer.trainable else 0
    # Keras 3.x removed layer.output_shape; derive shape from layer.output instead
    try:
        out_shape = layer.output.shape
    except AttributeError:
        out_shape = "N/A"
    print(f"{layer.name:<30}{str(out_shape):<25}{params:<20,}")

print("-" * 75)
print(f"Total Trainable Parameters: {cnn_model.count_params():,}")


In [ ]:
cnn_model.compile(
    optimizer=optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
BATCH_SIZE = 64
EPOCHS = 25

early_stop = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss', patience=5, restore_best_weights=True
)

history = cnn_model.fit(
    x_train_final, y_train_final,
    validation_data=(x_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['accuracy'], label='Train Accuracy')
axes[0].plot(history.history['val_accuracy'], label='Val Accuracy')
axes[0].set_title('Accuracy over Epochs')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Accuracy')
axes[0].legend()

axes[1].plot(history.history['loss'], label='Train Loss')
axes[1].plot(history.history['val_loss'], label='Val Loss')
axes[1].set_title('Loss over Epochs')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Loss')
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
test_loss, test_acc = cnn_model.evaluate(x_test_norm, y_test_cat, verbose=0)
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Loss: {test_loss:.4f}")

y_pred_probs = cnn_model.predict(x_test_norm)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = y_test.flatten()


In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()


In [ ]:
print("Classification Report:\n")
print(classification_report(y_true, y_pred, target_names=class_names))

acc = accuracy_score(y_true, y_pred)
prec = precision_score(y_true, y_pred, average='macro')
rec = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='macro')

print(f"Overall Accuracy : {acc:.4f}")
print(f"Macro Precision  : {prec:.4f}")
print(f"Macro Recall     : {rec:.4f}")
print(f"Macro F1-score   : {f1:.4f}")


In [ ]:
# Select an image to visualize
img_index = 7
sample_img = x_test_norm[img_index]
sample_img_batch = np.expand_dims(sample_img, axis=0)

plt.imshow(sample_img)
plt.title(f"Input Image: {class_names[y_test[img_index][0]]}")
plt.axis('off')
plt.show()

# Get outputs of all convolutional layers
conv_layer_names = [layer.name for layer in cnn_model.layers if 'conv2d' in layer.name]
layer_outputs = [cnn_model.get_layer(name).output for name in conv_layer_names]

# NOTE: In Keras 3.x, a Sequential model's top-level `.input` attribute is only
# defined once the model itself has been "called" as a layer. Since we only ever
# call cnn_model via .fit()/.predict() (not as a functional layer), cnn_model.input
# raises "has never been called". Instead, grab the input tensor from the first
# layer of the model (which *was* built against an explicit Input layer).
model_input = cnn_model.layers[0].input

feature_map_model = models.Model(inputs=model_input, outputs=layer_outputs)

feature_maps = feature_map_model.predict(sample_img_batch)

# Visualize feature maps from the first convolutional layer
first_layer_maps = feature_maps[0][0]  # shape: (H, W, num_filters)
num_filters_to_show = min(16, first_layer_maps.shape[-1])

plt.figure(figsize=(12, 8))
for i in range(num_filters_to_show):
    plt.subplot(4, 4, i + 1)
    plt.imshow(first_layer_maps[:, :, i], cmap='viridis')
    plt.axis('off')
plt.suptitle(f"Feature Maps from Layer: {conv_layer_names[0]}")
plt.tight_layout()
plt.show()

# Visualize feature maps from a deeper convolutional layer
deep_layer_idx = len(conv_layer_names) // 2
deep_layer_maps = feature_maps[deep_layer_idx][0]
num_filters_to_show = min(16, deep_layer_maps.shape[-1])

plt.figure(figsize=(12, 8))
for i in range(num_filters_to_show):
    plt.subplot(4, 4, i + 1)
    plt.imshow(deep_layer_maps[:, :, i], cmap='viridis')
    plt.axis('off')
plt.suptitle(f"Feature Maps from Layer: {conv_layer_names[deep_layer_idx]}")
plt.tight_layout()
plt.show()

In [ ]:
def train_and_get_history(model, epochs=10):
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    h = model.fit(x_train_final, y_train_final,
                  validation_data=(x_val, y_val),
                  epochs=epochs, batch_size=64, verbose=0)
    return h

model_max = build_cnn(pool_type='max')
model_avg = build_cnn(pool_type='avg')

print("Training MaxPooling model...")
history_max = train_and_get_history(model_max, epochs=10)

print("Training AveragePooling model...")
history_avg = train_and_get_history(model_avg, epochs=10)

plt.figure(figsize=(10, 5))
plt.plot(history_max.history['val_accuracy'], label='MaxPooling - Val Acc')
plt.plot(history_avg.history['val_accuracy'], label='AveragePooling - Val Acc')
plt.title("Pooling Strategy Comparison")
plt.xlabel("Epoch"); plt.ylabel("Validation Accuracy")
plt.legend()
plt.show()


In [ ]:
def quick_experiment(kernel_size=(3,3), filters=(32,64,128), optimizer='adam', epochs=5):
    model = build_cnn(kernel_size=kernel_size, filters=filters)
    model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
    h = model.fit(x_train_final, y_train_final,
                  validation_data=(x_val, y_val),
                  epochs=epochs, batch_size=64, verbose=0)
    return h.history['val_accuracy'][-1], model.count_params()

experiments = {
    "3x3 kernel, Adam":       dict(kernel_size=(3,3), optimizer='adam'),
    "5x5 kernel, Adam":       dict(kernel_size=(5,5), optimizer='adam'),
    "3x3 kernel, SGD":        dict(kernel_size=(3,3), optimizer='sgd'),
    "3x3 kernel, fewer filters": dict(kernel_size=(3,3), filters=(16,32,64), optimizer='adam'),
}

results = {}
for name, params in experiments.items():
    val_acc, n_params = quick_experiment(**params, epochs=5)
    results[name] = (val_acc, n_params)
    print(f"{name:<30} Val Acc: {val_acc:.4f} | Params: {n_params:,}")

# Plot comparison
names = list(results.keys())
accs = [results[n][0] for n in names]

plt.figure(figsize=(10, 5))
plt.bar(names, accs, color='coral', edgecolor='black')
plt.title("Hyperparameter Comparison (Validation Accuracy)")
plt.ylabel("Validation Accuracy")
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.show()
